<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 2 · EXPLORING AND VISUALISING DATA</span>

# Session 3: Missing values, outliers and categories

**Dataset:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). You work with a 10,000-row seeded sample, `ev_m2_sample.csv`.
**Time:** 60 minutes. **Course outcomes:** CO2 (and CO3 for Session 5).

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Is a spike at 30,000 real income? |
| Concept | 10 | Why values go missing; the IQR rule (instructor) |
| Guided lab | 30 | Blanks, outliers, floors, category rates |
| Team task | 10 | Three keep / fix / explain decisions |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- find blanks and test whether they are random
- compare three ways to handle blanks
- find outliers with the IQR rule and decide keep / fix / explain
- read a category rate and its group size

**New commands this session (max 5):** `boxplot`, `quantile()`, a boolean filter `df[df[col] > x]`, bar chart of a rate, `pd.crosstab()`

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [ ]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

# -- Load the data (Colab: upload ev_m2_sample.csv to the Files pane, or run the upload below)
FILE = "ev_m2_sample.csv"
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print("Choose ev_m2_sample.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("Put ev_m2_sample.csv in the same folder as this notebook.")
raw = pd.read_csv(FILE)
df = raw.copy()
print("Loaded", df.shape[0], "rows x", df.shape[1], "columns")

## Part A. Hook (5 min)
Run the histogram of income again and look at the left edge.

In [ ]:
def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

In [ ]:
df = raw.copy()
sns.histplot(df["Annual_Income_USD"], bins=50, color=C["terracotta"])
plt.title("Annual income (USD): what is happening at 30,000?")
plt.xlabel("");

**Discuss in pairs (1 min):** is this person's income *really* exactly 30,000? What would a manager do with a segment that may be hidden by a floor?

## Part B. Blanks: how many, and are they random? (10 min)

In [ ]:
blanks = df.isna().sum()
pd.DataFrame({"blanks": blanks, "percent": (100 * blanks / len(df)).round(1)}).query("blanks > 0")

In [ ]:
df["income_missing"] = df["Annual_Income_USD"].isna()
df.groupby("____")["income_missing"].mean().round(3)    # TODO: group by the column that might explain the blanks

**YOUR ANSWER:** Are the blanks random across city types? What does your answer mean for *dropping* those rows?

> _write here_

**Three ways to handle blank income.** Compare them side by side.

In [ ]:
options = {
    "A. drop rows with blank income": df.dropna(subset=["Annual_Income_USD"]),
    "B. fill with overall median": df.assign(Annual_Income_USD=df["Annual_Income_USD"].fillna(df["Annual_Income_USD"].median())),
    "C. fill with median of the same city type": df.assign(
        Annual_Income_USD=df["Annual_Income_USD"].fillna(df.groupby("City_Type")["Annual_Income_USD"].transform("median"))),
}
rows = []
for name, d in options.items():
    rows.append({"option": name, "rows kept": len(d), "mean income": round(d["Annual_Income_USD"].mean()),
                 "rural share": round((d["City_Type"] == "Rural").mean(), 3),
                 "yes rate": round((d["Will_Buy_EV"] == "Yes").mean(), 3)})
pd.DataFrame(rows)

**YOUR ANSWER:** Which option would you choose, and why? What does each option assume?

> _write here_

In [ ]:
# Does being blank relate to the outcome?
df.groupby("income_missing")["Will_Buy_EV"].apply(lambda s: (s == "Yes").mean()).round(3)

## Part C. Outliers with the IQR rule (10 min)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
sns.boxplot(x=df["Annual_Income_USD"], color=C["terracotta"], ax=axes[0])
sns.boxplot(x=df["Daily_Commute_km"], color=C["lagoon"], ax=axes[1])
axes[0].set_title("Annual income (USD)"); axes[1].set_title("Daily commute (km)")
for a in axes: a.set_xlabel("")
plt.tight_layout()

In [ ]:
def iqr_fences(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = ____          # TODO: spread of the middle half (q3 minus q1)
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Annual_Income_USD", "Daily_Commute_km"]:
    low, high = iqr_fences(df[col])
    n_high = (df[col] > high).sum()
    print(f"{col}: fences {low:,.0f} to {high:,.0f}; rows above the upper fence: {n_high}")

In [ ]:
low, high = iqr_fences(df["Annual_Income_USD"])
rich = df[df["Annual_Income_USD"] > ____]      # TODO: which stored value is the upper fence?
print(len(rich), "rows;", f"Yes rate among them: {(rich['Will_Buy_EV'] == 'Yes').mean():.0%}")
rich[["Age", "Annual_Income_USD", "City_Type", "Environmental_Concern_Level", "Subsidy_Available", "Will_Buy_EV"]].head(8)

## Part D. Floors: flag, do not delete (3 min)

In [ ]:
df["income_at_floor"] = df["Annual_Income_USD"] == 30000
df["commute_at_floor"] = df["Daily_Commute_km"] == 5
print(df.groupby("income_at_floor")["Will_Buy_EV"].apply(lambda s: (s == "Yes").mean()).round(3))
print(df[["income_at_floor", "commute_at_floor"]].mean().round(3))

**YOUR ANSWER:** What does the Yes rate for 'income at floor' tell you? What would you tell the business about customers *below* 30,000, who we cannot see in this data?

> _write here_

## Part E. Categories: rates AND group sizes (7 min)

In [ ]:
pd.crosstab(df["City_Type"], df["Will_Buy_EV"], normalize="index").round(3)

In [ ]:
rate = df.groupby("City_Type")["Will_Buy_EV"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
n = df["City_Type"].value_counts()
ax = sns.barplot(x=rate.index, y=rate.values, color=C["terracotta"])
for i, city in enumerate(rate.index):
    ax.text(i, rate[city] + 0.004, f"{rate[city]:.1%}\n(n={n[city]:,})", ha="center", fontsize=9)
ax.set_ylim(0, 0.30); ax.set_ylabel("Share saying Yes"); ax.yaxis.set_major_formatter(PCT); ax.set_xlabel("")
ax.set_title("Yes rate by city type");

In [ ]:
# Small groups: how many rows sit behind each percentage?
print(df["Range_Anxiety_Level"].value_counts())
print()
print(df.groupby(["City_Type", "Home_Charging_Possible"]).size().unstack())

**YOUR ANSWER:** Which group is so small that a percentage from it would be unreliable? What would you do before quoting it?

> _write here_

## Team task: decision log (10 min)
Complete three rows. Each needs a **decision** (keep / fix / explain), a **reason**, and a **risk** if you are wrong.

| # | Issue | Decision | Reason | Risk if wrong |
|---|---|---|---|---|
| 1 | Blank incomes (about 5%) | | | |
| 2 | Pile-up at income 30,000 and commute 5 km | | | |
| 3 | High incomes above the fence | | | |

**Becomes:** the 'data-quality risks' section of your Project Charter in Session 5, and your data-preparation notes in Module 5.

In [ ]:
# Apply the class decisions in one reusable function (used by Sessions 4 and 5)
def apply_s3_decisions(d):
    """Session 3 data-quality decisions, applied in one place (see your decision log)."""
    d = d.copy()
    d["income_was_missing"] = d["Annual_Income_USD"].isna()
    d["Annual_Income_USD"] = d["Annual_Income_USD"].fillna(d.groupby("City_Type")["Annual_Income_USD"].transform("median"))
    d["Daily_Commute_km"] = d["Daily_Commute_km"].fillna(d["Daily_Commute_km"].median())
    d = d.dropna(subset=["Environmental_Concern_Level"])      # strongest signal: do not blur it with a median
    d["income_at_floor"] = d["Annual_Income_USD"] == 30000
    d["commute_at_floor"] = d["Daily_Commute_km"] == 5
    return d

clean_df = add_derived(apply_s3_decisions(raw))
print(clean_df.shape, "| blanks left:", int(clean_df[["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]].isna().sum().sum()))

## Exit ticket (5 min)
1. Blank incomes are more common in rural rows. Why does that make 'just drop them' risky?
2. The IQR rule did not flag the pile-up at 30,000. What did?
3. A group has a 40% Yes rate from 5 people. What do you say?